In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
max_seq_length = 4096 
load_in_4bit = True 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit", 
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4080 SUPER. Num GPUs = 1. Max memory: 15.55 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [      # training matrix
        "q_proj", "k_proj", "v_proj", "o_proj",     # attention layer
        "gate_proj", "up_proj", "down_proj",        # NLP layer
    ],
    lora_alpha = 16,        # scalar
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 1313,
    use_rslora = False,     # if True: alpha/r -->alpha/sqrt(r) highrank stablizer 
)

Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [4]:
import json
import random
from datasets import Dataset

# 1. Daten laden
with open("/workspaces/Arch_PC_Assistent/dataset/arch_dataset_cleaned.json", "r") as f:
    basic_data = json.load(f)
with open("/workspaces/Arch_PC_Assistent/dataset/arch_troubleshooting_dataset.json", "r") as f:
    trouble_data = json.load(f)
with open("/workspaces/Arch_PC_Assistent/dataset/add_dataset_progress.json", "r") as f:
    basic_add_data = json.load(f)
# with open("/workspaces/Arch_PC_Assistent/dataset/arch_troubleshooting_dataset.json", "r") as f:
#     trouble_add_data = json.load(f)

full_dataset = basic_data + trouble_data + basic_add_data # + trouble_add_data
random.shuffle(full_dataset)

system_prompt = "You are a logic engine. Your output MUST follow this format:\n<think>\n[Your reasoning here]\n</think>\n<answer>\n[Your final concise answer here]\n</answer>"

prompt_template = """### System:
{}

### Instruction:
{}

### Response:
<think>
{}
</think>
<answer>
{}
</answer>"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    thoughts     = examples["thought"]
    outputs      = examples["output"]
    texts = []
    for instruction, thought, output in zip(instructions, thoughts, outputs):
        text = prompt_template.format(system_prompt, instruction, thought, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = Dataset.from_list(full_dataset)
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Dataset preprocessing finished, ready to tokenize.")

Map:   0%|          | 0/1637 [00:00<?, ? examples/s]

Dataset preprocessing finished, ready to tokenize.


In [5]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        output_dir = "/workspaces/Arch_PC_Assistent/outputs/SFT",
        report_to = "tensorboard",
        logging_dir = "/workspaces/Arch_PC_Assistent/outputs/SFT/runs",
        logging_first_step = True,
        logging_steps = 1,      # steps to record
        disable_tqdm = False,
        log_level = "info",

        num_train_epochs = 3,         
        max_steps = -1,              
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, 
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",

        gradient_checkpointing = True,      # Sacrifice training time for efficient VRAM usage.
        gradient_checkpointing_kwargs={"use_reentrant":False},
        optim = "adamw_8bit",
        weight_decay = 0.01,
        seed = 3407,
        bf16 = True,
    ),
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Unsloth: Tokenizing ["text"] (num_proc=20):   0%|          | 0/1637 [00:00<?, ? examples/s]

In [6]:
%reload_ext tensorboard
%tensorboard --logdir /workspaces/Arch_PC_Assistent/outputs/SFT//runs --bind_all --port 6007

In [7]:
# 1. start training
trainer_stats = trainer.train()

# 2. save LoRA-Adapter
model.save_pretrained("arch_assistant_lora") 
tokenizer.save_pretrained("arch_assistant_lora")

print(f"Training abgeschlossen! Dauer: {trainer_stats.metrics['train_runtime']} Sekunden")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
skipped Embedding(152064, 3584, padding_idx=151665): 519.75M params
skipped: 519.75M params
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,637 | Num Epochs = 3 | Total steps = 615
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.775767
2,1.954168
3,1.536563
4,1.456562
5,1.469119
6,1.512143
7,1.321329
8,1.357432
9,1.291939
10,1.120632


Saving model checkpoint to /workspaces/Arch_PC_Assistent/outputs/SFT/checkpoint-500
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--unsloth--qwen2.5-7b-instruct-unsloth-bnb-4bit/snapshots/b849156629a5c8d6e9f95ed2e4ec8bda3507938e/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": null,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention

Training abgeschlossen! Dauer: 2040.122 Sekunden


In [2]:
# XML format test

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/workspaces/Arch_PC_Assistent/outputs/SFT/arch_assistant_lora_1",
    max_seq_length = 4096,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

prompt = """### System:
You are a logic engine. Your output MUST follow this format:
<think>
[Your reasoning here]
</think>
<answer>
[Your final concise answer here]
</answer>

### Instruction:
My Hyprland won't start and just shows a black screen. I'm using an NVIDIA GPU. What should I check?"""

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 1000, temperature = 0.5)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4080 SUPER. Num GPUs = 1. Max memory: 15.55 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Both `max_new_tokens` (=1000) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/workspaces/Arch_PC_Assistent/.venv/lib/python3.13/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/workspaces/Arch_PC_Assistent/.venv/lib/python3.13/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.

### System:
You are a logic engine. Your output MUST follow this format:
<think>
[Your reasoning here]
</think>
<answer>
[Your final concise answer here]
</answer>

### Instruction:
My Hyprland won't start and just shows a black screen. I'm using an NVIDIA GPU. What should I check? Provide steps to diagnose and fix.

### Response:
<think>
This is a common issue on NVIDIA with Wayland. The black screen often indicates that the NVIDIA proprietary driver is not compatible with the current Wayland session or that environment variables are missing. Steps: 1) Check if the nvidia-drm.modeset=1 kernel parameter is set (required for Wayland). 2 journalctl -b | grep -i nvidia to see if there are errors about modesetting. 2) Ensure the correct NVIDIA packages are installed: nvidia, nvidia-utils, and optionally nvidia-settings. 3) Verify DRM kernel mode setting: cat /sys/module/nvidia_drm/parameters/modeset should return 'Y'. If not, add the kernel parameter and reboot. 4) Set environment variable